# YuE2 on Kaggle — recoverable full-song generation

All computation happens on Kaggle. Select **GPU T4 ×2** (a single GPU can
also be tested) and enable **Internet** before running. No local GPU,
public tunnel, Gradio account, or API token is required for these public assets.

This version uses native ComfyUI with the official INT8 YuE2 checkpoint.
**First run a 30-second diagnostic**, then try 180, 420, or 480 seconds.
Duration is a **maximum**, not a promise: the model can end early, and a
token-limit cutoff is explicitly reported. Seven/eight-minute synthesis
and T4 memory/performance still require validation in your Kaggle session.

Native YuE2 uses FP32 text generation on T4. The selected GPU runs the
stages sequentially; two cards are not automatically pooled. GPU memory
observations cover both cards. FP32 VAE and tiled decoding are retained.
Cover transcription is deferred until this generation baseline is validated.

Outputs and recoverable stages are saved under `/kaggle/working/yue2/runs`.
**Download the run ZIP or save a Kaggle output version before ending your
session.** Files in a live working directory do not survive every session reset.
Downloaded run ZIPs can be attached to a later notebook as a Kaggle input.

Sources: [YuE2](https://github.com/multimodal-art-projection/YuE),
[ComfyUI](https://github.com/Comfy-Org/ComfyUI),
[official repackaged weights](https://huggingface.co/Comfy-Org/YuE2).
ComfyUI is pinned to `bc5f880dbdd43e426077f42ccec53bb577ffee9f`;
model assets are pinned to `2f76ca75e6ee094169de899cc7fc99d6887e2196`
and checked against the published SHA256. Model license terms remain applicable.

In [ ]:
# Generated support modules are embedded so this notebook works by itself.
import base64, json, os, sys, zlib
from pathlib import Path
if "app" in globals():
    app.stop()  # Stop only the server previously started by this notebook.
support = Path("/kaggle/working/yue2_support")
support.mkdir(parents=True, exist_ok=True)
payload = "eNrFfQtz2za28F/hutOJlFK0JDuOrUQ713WdxrN5eGKnd7q2h0NJkM01Rap82FE9+u/feQAgwIes9OvO3W5rkcTjADg4OG887dwHt7eR8FeFGO76fhiHue97y9XOyNm5pn/+RQWcJJ3eiSxPgzxMYmeepE5+J5xlGMdi5sTw9kE4J8livvp65vxenA6dcLGMxELEOdXwuLHreMd1rD6nWMePk5nIrG4/BnFwC01PiyxPFg4VGDnLNHkQcRBPheukYgoPaTAB6LIcCmeuE8RQ405M76Gm+LZM0jzzruPr+CyGElEEb4NMNsl97hqwyDfGLHjOR3gVOUF0m6RhfrfI3Ot4CR2HGc5CJiIxxdFBx3kOQ6WfWQAjD+NbCUwR3+NDKhZBGFdmCmGT8zJPYZS+Py/yIhW+j9MH0EMbccIzmOE45Nt5EU/zJIky/SaMsyXAop//kyVl8SST7S+D/C4KJ6rxc3jUhfJwIYwu4mKxXOFsxUv9LkuKeDYPYb7hfTYvqyJyyC6yYC5gKrIkzTx6rzqLkmDmY2WcoQdBP43+CA+8BU63v6C1R9yRjfLHXHzLfRFPoQw0jium2kaEuzyltZI1jGX1JKKsVOnOdezA/4JpAogQTv00iBl5ACvgEafOVVjkT+/CaOY6sxCK5C6PYnkXZISA8JtKc4M0LPktuwuGrw54vNdxFwd6cXl8eeqMnac1Pl3HMzF3bpeFD5iZZ53uiBtJBSBA7FzxE/7v6XpnJh7CqbiG3RG6zvUOYHIyDXKA7jac4FuaaG9azAJvIRZJuvJ1kU7YdXad4cuXe323bBMaSUUm0odNbagSrU0sRXDvbwQm+OZvAGhdtoYEJQQsdmgxOkYbPHYfFivOO90uV7kpp/ARNqagSSzKWTSWskPzfgWTGAIiQMOr650bgOB6h4mGR1sFyJIqx6+hULfsZCJuw9hPi7gD//q41VRP+BsWFRvxEDmysoRc0CTJoQDutU6SeSJ+CNMkhn5+/3o69L98/eR/+fz5ErsDRM2S6EF0ZE0NMFS30LGDbbrUNbSD/YUzhherhXOgloDlZfVdWu4/CsBgOdquF2aEnHrGCNQgzITzWxAV4jRNk7RzvfOliJ0FbBNnHqbw34lwpoD2sIzOZEUngDweplGI23VHwgC0F/qGUuNyl2wGSMNOE/qPsW7iefiwRjgPp3w0zcIsuE2FyJxHoNgEI27MmSM71H3RenvTSARpx3pVLGcwwhLcsf7Fcz7G/7jyzBk/EcYA8iHu02rEQO8RoUpUGsGmX8s+tsZ0Y+BGIdyUuU87T+4sph+hbN7eDiUGI5bocehVl7hC465NNMwtHgpqqk+/iWmRC5pRaK03TWKkyHQ0A2LAuIRzK2LBLIKeZknQmnZhCV0yIVLjl/U7cODCngsiBSoflCfBEg9IdY7CQb5c4vHKA37jxAIoPfQZAmwpnNRwgsJ5nyfzOZ1ZxXQqskwzI7SBw1toDNscq1PU0+9KKBBSLP4/+uz1HtNgmZklaNOWo5l1gEGYu87LIL2F4+Xly/tH/GUubMv8G/Ommm9pq6wzwcMZxqBh9yZhPNuumhcsl9HKB9iDIsqzjvH9AXdbBu3OwmnekaXT2wJP56xWzlsmS0AV7FQjAP6PTkVohEsBGtALQgFV5AfnFA/p/A6X807AeoSwYNMgClJnIfIA9mTgINuZ3IvYOfsFhsOLTTyDo3iOKkSIcvlqiX3h5ORpp/begFPM58jRPSCsT/Z6wLGrho0bmltxq2VS3jNYQp8ngFLQy5Xx8caoZxyC92JFE42sRkeDYoAH7GkexkznxtUtDVQVJv2JZnbt0zSpw83Y2SKDMUDlCnTercg7xBTAZx9ZKGv9YHulTM5xAN5SpHOmUyI1kSVPVxUsBgSXXSI33KFTkF8gvMZ4vDiAd43HUkkFcUgu9AJbMGBwSn7subZdnF6XMbFbbxwqwjRSm5dpIewCIsrElgBtvV8bOn4XQD92iZKn7BgDkmOp9V9p3uSCNiKLxjXNDDVhnz4hTSbJ+k3N3dT3Du8NBhe3RiTiDj918ZjU8BP7qB7cpjZEPKPTdaTa86NwEcKhjphWrgMuGJfmjQBDampNSgbcsVoIOrjhQzwjWBsQ3umpDdHYqqQrozoFwqaB5cdvBudfaWRtP1YP9MrukpR1jPRpAjuZ9pmeiAakbTrba4V4IMc/nyD7BrznzAnzTJJemnB14Aexw0viOZe0oM4jkm7iuN44JuWxmw5j5CMz5iawn0kxAxKEBD5DzUE4Z/YSJL9lkoVE8WSXKXQoUuyyqfn6DOEBy8i23WzAviAFxjJNZsUUtRuJIzeKo1C423hW1/ajcbB8m4pl7vwMrZzSTxwQcCTwfvTMDlP8KKPA+HpnHgCFnNEuRYDHeKhBO7CV+ItPVcdNlG4TMtFUWDyb4mPqfBruDloUWIYap/YXWCTg1Obht5Ki0YapMEqNTBKJ7kBwKsJ8x6Iw+MA9dLcgZ7qpKFglRc6MQyNB4zZ9m67JjnCrz1M4d2oEr5EQEdREhVgZ0VSImK8AOf6SOl0FKEy/IqkicB1fCxbZ1WB0c9PYDjLuYVyEOYr4fxRwKGRARTPgoCRpRWi5ka7zT2fw1+nTErZpjhvqSoqJuLFunGM5w6wdy0A64t7Wb3jhx0/0Z40oPo+K7G6MR3LX5GKaeONmJNqA0T84/xJi6SCSAoHPnHAmgNqgIs95vANKB7xo5iykIhLYzCIippTVSKhLvA3CGDhOSWSAfgU57EVDG+UpiUYgPvhSK6WhgPHRma8wvbEmMhV18aixaHdDM161d83pbAJfbwZzx3//WBqbMQZm0ZPna3a37aJ90PjPNAqyjFqBA+CExVlFw+gbyB13yaykWmefzr9e+pe/n59edKaRJclJBHtixUYoeQp4nEbhEn8Cd33y4ewc5olog9ISyU8Xl1/OPv2KH9drJWp+Ob38+uUTdwdwGw3w93dfP51cnn3+5ODhL1g6V0fiCdC1Xz9/+Z2+BcUsTHZJ3c7TgaXKUcmqchchvK5TVXORqFhXgtXG36Hq3er8nsP2AgoDB/3fPr/E9dhziK+BIE7FWbxEAj6iNV/TvDcsx6YZL9fl75hzLPX56yUO8tPnX04NZGxeCxgbL0iDzgC+eXDyh8uarNSgJvuUEJOVTZGBegS+QzE3FivTICTA0Yh1PJrlrsfEHvdJB940LD9CXFv9C+AGPwRIV7O/ffnJ2sJHJ4zzAyzBp8vnFrUs9X+xqBJiS79HygvUT/C3K2NcN94MCPP0rtP15iDt5vB3uizwv3iM3xaJfeyiGESteUAFF6hI3SOtSQGtdeSX7C5YiqvR8KaL3zsD1znY3wKJTpVSN4n5JHciWlTnauAe7LuXNxYyac4bVZdhNkejmoKg6wVR1IK373CQQLrPE+AdVNeaY+BpSU3mPO5x2xKa7I20/LFysILiaAFrVJ1c78jqnmHBMgWMHHmDNCBtPLbioXrZz4o58BwAIW0N4p+M3rTeakxWnOQxRoWW8OkM92WLEUuzg6Nhn7XGSnyU+Ha9Y3Bc2nTWsXGfAV67pOHSoALFUyCM1Q8DvCTzUrGMgqkoq7g0OKPQZiWCnjY02oXzUqHv1vVnbBNDaA3rWIe6c/kzjxt5UQtVu39ZzQZF1bixrPptqt+MoT4veqnBknyLg+QXPnP6Y2t/9QbQP4j5Y0PUNzpr45ubDaFefpcmj75QsqMfzn3YA6jNBqTzSd2dFku0rzVQZYkodcr8IQlm/y3K/Jc4nL+XNLeR4O/mbrKkAHZCmfEaUM7Um9YteUQL5P6wTGKy3Q37yKao6qOF5fDA6KI22A2S9U37ol7jeep/QUY0CSVreRbBTLCVLXBm4XwO7+A0IFPlEql3E+Wtj7iN4KItsEoncGDGHPxl8FlRghqmMGWAYf0scCVASsVMICBtleSKbXVAEaaozus2HOWyAesotwZXJ3P10bHdAtr7S0epNQFn8UMQhTNpDJUnN3WB7bOq0l6wFMYIx669aJbR3BqPKt+qw08i2x6sKnS3VIA93Y+wjav7G1J33KOuo1OWI8+Iqu6mi5BRSai5kdajmlagb4mvtcJqP0vDEc/CFiRckd2mI7rxgMeXm7gD5A3WzXz1Mbbw9wtVDBcR7uOvv5x9fo5u60L/J2ISNmKO6xEmBlBkgYpB/AZrq161M9QWMqviTVy0/lbno4ffy0ZncG6LpAQYWOlhlZOWSiBUpanRMFohplQMmGhvw+IAz/5hv99HqCvwAl/ivKVS3wFrkDuRCOD0IqBJ/+gkc+jDuX//pyPRxbRUMterur7q33iXlsTikYNZpy4wxMuSxBG/+r2ywm/Hp43iAQG5WTgQwT0AzVgBcASTTIKwCL5ZzFv2RxGkpHiGc6OjK/DrziRKpvdS/ziG1/QdRCwPC3e7zTYR9gyhqkiusPs0DVbAiURhTmAAMw9gDFifTXA5u7u0kF2z0XSBcHUUiLu0Fl4W/gkHzMuXTt97ZaMLFH/rDETv4Hl0+IUI5IynElWh2kwYrZwMaH6cv9FnrPR4DB7grEUnTWuqf3DOgfrjEXR+8hF43vgBNUToWUr+flFIbh1vyMkzpYPaQR+FEDpHpSuuYhROwzxaGRZ/+gJHaBh3Bl7fhaEeHR3BBOC60kFAC0y2QfhusyY4Z1c3tkMam43ni5xOmo6kVrCcwZRp9rsPxycwLNfR3wDh+dP/Hv8GX6po+3cy9xvFWQTdLvpX5FeipcjfZXPvAq0P78LIkhWBruJwpVQOCz9mXff0LohjmOjx0G3DdmOiF0E+hlmGdooJ7ZnrHcAKf7iP5ze67eTANCwarHe4SGSJLf2p+sb2cHlzjJpB4FZZpUXlr6ipETf4E9W9cV4SVlXmZCvBmfZXPE+QSMw9/NVpKjGnQh5LkEi3y92NnmL4rZxe/E5/1Sc10fhhuKW1F6k6ksA74MRE6swS6BcpLyzD9M6ZmXu8hg20VdBdSKBvkeKHTcoIuz5GTc14s6+AdDhRxW0e2vCnUAVkecNVgHa07i7kMXzC84l2OP4yzqMiVW4zBvWkudxC/q/4ZLA2QO15Vgb4tJ3Y1azJG6HNEG42QX5bQeTD3FabgOGRdUV5NpjV2ADvF/F9DDwk0qOKGwiWRT9FNQmmAVG9Iw2MyVWMHGW2UjiG71r2M9Tlrctsrdq9rnISZtO9z2cvlsG3ZAZZsNVzkbU1zHV83IXkNwF/qVlERGoJf7RVhgMzXBQLX/p+om28HHpdQgap5FaKx8E3X02N7a9ltR+LIDUal74o5aw6/xzjKfQKyMh39LaudFdRIhlfv0MYISxQMgSy1f7Jh+OLC//j8fn52adfkY9/AvHA8308O3x/BKdwRhQW/5KopY1upbnMrZh33KrC363qmVxbdEHv+HU9LET57FsxIRcLYAVR9kbqI+LpqjdPBTAGaR7Og2lOri/a2X+Z4mIAa1K6e35XjMVdkGGsxP9PRAUcDSrK4fjL5dm745NL/7fTLxcsGA1Ktwrp8kdCuDquONBhpnzLZ8AuZ1wCzkncEfdilZFhHJ7FMgDsSVKiSi5vvBFxJuh3/whLGo/ZOmuZw+UYPdajdGSXHv8FTtW7E98kbIYrsakRxvG7jgW45ElIU2YceMRzAJjI7yzugVvp8IMag/gGGOQn96atfz5zLb5FzSg0kOFv6WwxLtmXHo4dGh8b3cnGbOdEYmvgHJ/PEsCmznwmmZl2hkMvg14FKuXC3gBkzIHVqa1L8+RXGBDycOjUOYx5toqnHVUImevEEh6eZULm6BsRmYOmNSmtA14RR2F831mEzGiWk6+9xktlDbZrx6gYMQ/lYntUhSyEXaOhmiZPN5UVEepEK6iovNlplXCBGNHgwJi0rZElPoXoqBcFi8ksGKmJRsg6h0CKB/3hvvwDO2SCxKEmXxJc6vynZu2tIws0bpCmYA1ybx3VQ0Lwd01fLLcQfZTMfK2MFJcJz8MMDqGIXOL9POE2kUXkdsbU4eY4imNFQ6kKRXxMBPrDhzNBvn/AiTiG7749GUsie3qtS59VXjXtkt7ksqqpxgpRyXa1ut6RYiGeqVUSShyLapmsSrobswXDO4sUri1up5MkiTolVLKJdT2ciIf09PKlhNi1LFsSGeS37trACsNVuTYt7Kun9wTJvOPq7uOvSltN37Gg8rmXMLjE+ZZIosv/Q7uWcz3CENkEM9d6skmfVZ3vTQgELO9JQjIqNlrEWbGUPlLKc1Kfy+q0NmN+LCiMNSU49PMzALCGX/UXoutHBsKNIdqQP5fk1na1NUQdD+y+OraBqTh8yk0XArkETitG0isRGpGrW2rqqS30yYtXHeSLO/BRiiggznP8LNIpiY7PDE3p7muTWQEPThvXuQtviXiAJDx4NTjY3+s2+gmTINGBEoev9qDk4f7BcFCOEuFGYKFF5y0cvM5bbrcKOGlFcXz2rBlSmh437rC/Os6HZBpMigi5AOiQBR5HhV3ZtEgtCAN0ZQFjRBxVHUZZ/lZOhL5qRQY5jYf7r14f6Ji/8E9UyHZUBFTPrgXPe6SaG+rplNL9W2fAHs5QH35vJsnnwF8uSe36QAK6BlkBVR35VYdUFy4pwJQWA/tyZf+wFs0aEzV4UhLeNHDjvFcsZvxTkgtY0vsebGZcDStA3HPOiAfWGzDkAGwg89N7FK4dDQf65YXT7+bSqRSK8WTzEJkulmGgklt+KkOk6Q/yGNoloJG3X5g8/FaMPjBhOdkWdKD0XZGHkRE3DexAGZsNorLU7pWvVlklEFs+FGmEEJOjd/WlCmjUr4twpqemFvVsRTercOYyhNnk0LCRk88f3/3uw38/nl2SpWYyfTU/POzPJrPZ/p7YHx70X7+e7w+nUzF9tTeZvIKnuRBHc1zCjyBefvC/nP52dqGMQMP564Np8PqVOBCif7Q/ODiaicOjo+n09Xx6dDQ7ODx8LYaDowOsfvL+9ORf55/PPnHPFJu/N0E96CF6dz6kSV41GnOPF++PYQxU6ehgLgZHR3uvX+/1j/r9gZgdTmfDg+Bof38SCCEOp9O9QTDsT4LX072D2WC2N+0Hr8Vgvj/YP5pNDoxGz/6NJqi9o4P+0R7MQP86/vzzxekXOBabpblpsliAKNqRoT9ARB5nYzyUQciJH+SvKQcyji2fWVtEkdu6RBcPvROuUOv2jXfyN9zF2M0NdwL/ch/wrwxhZ0nkOSUs0TiWWSRgflLkcIKO5aOO66UwBgOkE0qrcM5PRLrqYQ3IhXAzoyb/cAxe8LJ8BkhO1J2foH98Iub8qnfQ7/dHMEpEzzHsFlm+W6GgFcklD8JIMlsg6IhsPHz1jJRqctUkkWadBoPp9Q4cjnA23jorAeKlNFn+4FwIce9MgMQ9BuksQwXancBEGUFM3B9FNiZFmqHpDCbwFlh6+E0TjWxB4nw5/ugZkg+BQeLPJsFHijcZdI7EfNht/oTGo76SWr1coFUNjqoBTm232zDGa8xO4f0nodOklKC6HquIO0SU0CtKiqHoIErWKprrDqwa/RipQPr/0TQZTtvkTxEraZMt2hdJfHuSxPPwVh2z+SoSJL1J7maVhtPMeGGqzUZstsON2le2HbSzUHGiCPMiitRSAQtUfjBe+jBTsxGxaLClZTNKx2x/fF3JImF+fF3Ch43K6CP5cb8PVE4OUCz1671hpbfp/LYclLZXwbkifDynVb1Xg6HxBUl+FCzVx4N9ie1REPtJHK1Y0CljDUuTuu8vkyyXuU/IsN7gdGwwvljCoyUinNK8b/l+e/dkIB8PKGgGvOa4N4D1aXInrQLAONEAAX/4fhAw4CwSOSnVKLdLLwdGCD1mqMEmmBYsgkubNfVuYqaG6xA56dpnfLl/2H8ewo+syy51y0pKP/Q8aEBa5TOnUyyBCXYEsOs5soJFDpxfFW4GAyP3mTsjl3naIOwzGSWzFf8me8Z6iwkEFKOoDmpVwYZtohMqtofzQK09v6zk+K7WlOQLeq3RuE2oaNNu/HyiISKOHTXFujH9KaDmRBC3zxYJUGzioYBmBa1CNOJrbVC3A09B1COIpDsOTR82KSdwKefYgo9YeoE2oriEh8iRKzejSb/kO4tsdetB23LO4ZsptkpE7jMiQ5dvMZHLwV6r4dF2JglABqdqesJjbFbcCrRmOn3Po9Z6g+r8lxhgk9Qm4PaG1jYrC+NrOuy2WxAZkNoM6t7Q86itdkiJuDcBWJIBKoJPB/tbwKRd4KlWM1xACg72t6RR5kGjadSgBM74jC+HW/hYKmPsybtfNYADWNU3UtYjpmaA1Al1hA4nMKmC2ynnUB921Xm0vzo/wqTKARzs6xGU3xH8/v5hHUst6mIeoO0dygLYZ2N79hJbVd5W4UIdwRb04SvGSUOlXdXQoojyEP0KkZE8fOOU73nOJwL1NndBNCfPGaxLs6zTGRVhNPP5lO2gGZIT25CwIp0gkQzZwgnL5z9jVcS64/Mz5zYNlsBbUzgSeu4AIgIZe0zS+zkCwK4jIPejlzDtJpklxMy/orytbUOXTLFj2VMkmlCnHFEx0DF2wED6pf35RLsgo+FRpBeUjI7PM1YMqor3y5zsnfhcip3r9VqLA8Z8GCtFQFxd7xyo8OAaDBWzZ0PnpoO8+q06rmScUN0NN3ZXmmWbhioD365o2lxncFMNQ6zCIHli8v+NbykEzDoU8SUdiv+wD8XN0WlqKHsbh/KrjFoFSlwfS1NMczm6IVbo33D+pVXEpgGEVTKr1zuKlZPvJQ/Z1CyeVrpceayy+7z+gA8uexsY50352Xrd2A/a5HC80pLR916TgSJZ+kt+PlLP9/hsJ2Iz0lksRU6hrz7IjEGUr8h52Ov3X7EnBr3zH8N4ljzSp361oXUlupvXn5cLZ/VZDKis9g/OO6SehfTdo2DCjIgS0Co4KSh5FBxrwOawWR9IWkzGJqnk9KrtHasvrOLH4pSJkl0QBLCQ3ChMdYF9hYCMUZZgmxlnMfOaEXJ/I0KWLg4Ne0vGoBLX2oSLrbO63zarpcDWmGmCYK6Th1db7amPOPPP7Kr/wo6qTpO1t2wetVqxttta6Y90j9ICTK0ly9PG2KCmH1RlN5Ivqb0bX9Ua1nsT99SGvTgkDJnf+uTrXx8/fDKj09bfc+ScwnG+onOHzpzjMrjAwlYzfQThjDoMJmglI86EYG3q/HVr5/+6YC/FZxGLvF/Nk4gxi9O8PAgTLP4Si9ug4Uu1XRmKFy6CW1X0wEBci5Kb0g+jNfDUBlrDU619WBkDG6RPpmYerndmy4U/lHn/MGVOQZOBX7LbhV/EIYcfUKyHiBPUVsrm6uutpvtw44Y23KyalrkMPeGpqHAzaq+8tmmU6vqotevfjk/ZEZwQ7FIlo2lZ9Cogh+WaPATCxIMhvdTcsV4O/YbUEczq2h/lS2W0r4xk0H92Ftu2io6D4Qkx58mmxGVgDmc6wYZHpnLN0Ku55I8xvt6RIS+7yDCDVE9hMNjJS44b7ZusE0kOzzqOWCW3drPStchigIr3sdEfRn3JfMVmxBcVeBDxQ60wvqyVXK7yO3K/Leth2UkY7/KnWg06sKtt48vGknKwG4eHCvtqg5QWJMVPtWZhDWTYDfyqtiXtH1DAdjSmj5MgE36RRvqrEbnE+V/R7b+mZOXJLJPEetPEDpGRbhUyHo3QwCeLne8bqKC87CzvN9k+4PH575fvP386P758L0PbZKM/OZifltxpxRKemkqHNABp5Tc/dRU1saR5bqKa6FaF06mlq1V4/85///Vn/5ezi+OfP5z6l6cfTj+eXn75nesStWge1NdPP3999+70y+kvjUXllkX0NFZEmqbl1sQsxZFfxrCSuc5cJJZdUSRH1QtMOflnMwVaqQSkxRLowkxlx32RoYEXuArnfHWJdvLdT8XifGWJwaamUeGW6dOOgpX50VsmZMCBMvi9UYlQcfu/AB6FWW+Z0ohxv8yrNgcUovSb6PAlp8VO5mpolNAIpwzfHql3MpT3oZ8ojItviBLI+UMp6VbkUzDEW6eD7ib97jYQ0yw7H7A9lWj4nOnIr+dfEQryQHA68uWeN+j/VFd1kzHee7wLpwjdbchhodvP268hKqiVTpb0rEbmY8lvmArZVAhMho14zn3PwuzeLzIo2NHUp+uR87JMhW1BrBt46wyGz0MIfO9nPM+fVL2RN5ivnV/Dn6mlN+yGWobwDYb8jdRaebEk3GIrZKN9wzoamqyitfNDEuNuLfiJ7OJXchH4tFW/en9QwgY0Y1ttmakRipQizMc1m7jRYioeekD5MqnueX96/EtzwzVTNx+Q2hRuT4Ts22MagtyP1nV07LNzl8SMMIYjDy2iLfPVNBlzAfy3nI0eiMP53XjAj3d5vsxGu7tQ9K6YwLmw2KUjufc5vd1VlwnIdkzPjWcmtA0QVbSEBQNmZczb6eXJe799Wk1qHhkTxyZ9bScBmc0EdJt9eIozieRJ3TOhp5yTD2QcPGjcS4EB5kh63jgFUZJYPBLvZYeqhmlOKdYbJgIbVpOAnD5ZZ7ZGLEYjOXILm6jPjcbJytjVkGUk/h36WMxCTHbDIyPSzY7FptcBHkvSMx7nZdG6vZkHexZZkZxz7DVGdNK8SJGGmT6eKT7seiDPiZ5y+KIwQcX6mRHLsFRtsZcywJ+vl+DUOPiT19R4YUTRU1SxEsJC9RPeTgL+GUUPiyjEAKfqIOvJixV4Kt4NM+YiXOvx+KnuTabOuQ65a69roXPSd6ah4jnP0ackf4dBlrTmTaCQN1s1DVi86nyzz1+ak/EYTzntJITD2ArRNMeCJ6xkV/D0kzECb+SlJ5RWnamxdRTXT8NpEqNXIPsj2Lw3iz89AH563yvLZV6OGuymJswEZJaDCo0PmFb50oo0/pQ4vZ7kyDDNbx7IE5yAhaGCEIKepYAweIfLPJzunnz95dghuDwrM4/cBeWOMbfAEhVmfKgR6yS3Q6pR3zoiJDtBCcV5wE1eWlCfVHXlFLDnehBnyHgBxo3H+96r196BPCiK21tYpnkwFT04Kcbjvrd34Cm1hLq/BV8PgFtqjmxDBzKC15RVulZmkGSCDrDpixcvLB9KV7tCuupSGPM6GdehPemWF8lgIEA5FJfjo/37EM5BEdv3y3hx7FFiWdhiQYSOUO+uY3T91AXofoQw83XEeYfc/E9juiJIsmuIrcF0KiJ9T0HZCmV3QHnPdK/jtn1fcbE++zh1Xvz0ootpDeSTB0+UCeKfY6czdJ1D6psZWJNTZKOY2lj/HA+9QwSB88lkYySD33tDxHJsFAHRzJfFYJmWMK5QlLdDyG4UQXtCXJ0JYNZHoQtkknVZS86XDjQ9DRbyVpellyeA0/LOiV15GcwOHHTBJIQJQO2mjK9oAqUsB6BQVMQfEmgY4yzudAbuvnvoHux33TKzDrQBeMtZFLgweUPtoYPbfbX68LuqP5jV/Si8Fx2M77HQSVvL33mkq535syT3ObNE7usLnzp/uPfugysIy/zbPwI+9WXaCrwuCjHKMG3CpCuNx0gLUPxGIRnGeKpjb1RDQFcNTn6SH2jSXXVZEGqw5K817l3aqmoD3wXp7DGgKy+MIK5W+kYUiDb9jdtCHZp5HsXhVNwCBzcN7jWobfknR2cr8K6MwdyUrjVY8q3T30owwu3+pGqt8SAr4uaMFGaMj31MKXB0Zjv1wspuJf3at5vR6omBW0dxTzIhAnd20zyxFWmTwhFsh02KviGOZXn1AvfzixtkW+BBogs8X+8QmVsSgyAH0DVOUSMzqD0jobpOTbN3fIJZCUIZMANSul5NMQHWWbj5RjYTEL4o7rsUmyyCoyqNog8rWjMK28f56WBibPMuOgqfZZgRRuNaODtZQZCiZ0PTsMjSke2WyiQYCnwo/QyqrXxvfKzKJFbRWVVFcyt5XbVwow6ZJDcJU1s6L92sLNfYTO3eDExzFZFThmYgSGozGBcfGBcVtXA3xydf1bIDJpi0VoooYYlEY1TDGyItL8Lu9c5PKMgA2A9XaPlSYuJYvx3SvoM6mFalfL13UyOl9QneTEbL8bsGIriOHTLxrMN+xSZAZKqY9Ahmohr/DTJNw0RZA3NmATKSDxMI8kacBNJoI9RXpkwzClGExlYSr14wCkvalbEdIIdQMN8bFVlU0CUSYUahATV3PEZNJdaSv5fG6my1oPDnJvfHCkpTHCIPX7/bNtuKMRAKOggidKNf8YbOqqoJdI5bypv3nNDWVDRsJwmoHAoG/9ay1MngLWkAIxKHKMqJKkwdjPJriXy1IbDENohZTSc5slabzHacfz31jYDeahRNW9u6bnMC2e8h6N22PtQ55tdCedVJSqeZtUO6xOIrZgCrqN/rbdgKFVSnuAr5bMlZFdZRtV/mHOjWiqMbJip1cT4wzzemhhAzjwSf8v4A0iCRA7UQsUNRl2THQBgi9MNP8M5StKtVckJyF5/4+jVKAl1g9N27870hu1YbfcD/L/e5Z31VKmUNk8m73lBUYwEzFKDX6DJJ6N7UBPbfYwJVK52ryHsZuGuYcGQknDThcAJEJIEz2nauCi6XoVcvXYqxA8I33utXghxs8913GVfQwZUIkVKGvMiqlhZyP2rKk0cmHozSlVHwsmM2qhkIoOLKdYoOc3ZoDqAxOzzQ+2JMTjm0n4zpofzQMsMczd1YTuFWp5AUKSiZFF2HSD6Icd67LNNcovsX38+4y+i+3nhrGMnIlWHAI0VCyedyDeVfio0CwrlM4sbLuiboQD/WJWQsU9N9XNXkFliTYqjxh1Zg09I8rWtX/JgRm977y8vzlqC4dplF6Xqf1AqtR/D729TDJV/zTzsSq3s12u/3+8jRd1kJDmXMDUL6Qbk91LwNDhuR/28wPsq4RaJ2EvlAEju9ZE5sl9XDnM2oSSG9kU62XxHXuC3xflDTkrmLkdoRhcbUNyP7K1Owrsd/OhxxB79rzO/0ni9YhE4Gw9deH/5hb6ZqgkdlsccacFbjXzqeusB7tvsNzNnoM9rd1a2PnrCptSkOYewnKs8rrKY2P7FEyZf78JMFK37kjFwugdnoVYrmH7oXsqeuIXa1HC7NQ2GG4nQPSXkvCorYsGLJT3AI5XAu9KRA19aRaiiKeiwT6gr0/fEOpEscTcPXBrGxpRdikqFqLLTIvdwb9sgbiZ/hSOvRSd+bpkmW9bSypzqsb1Jz2dqX5OR6qEqTC+C9Uq2w3bFnpFVxS4eWmypeSg8WFQ8abMpVVHFRMcyl51R9UxCxbcv6jiOgVXWMoVwoZ4xVxiQO3h0bYF1c/vL56+V39kZUzQfO2c8wVWYSV2XjGVBJZNbUvZWLBOhLEofTDmo5JCE05vkO7zavlXyr26kLCi1UUVHOrWUEZd7jrLQY4yczXKTAy41IG4MpKXVQs8KGKrFpNmJtSYrLs2HYcDCqXJZtbSWT/wAak5dHzfalGJSQL7p82hyi0RgA0OTBvBldNjnE2t6qzR6Nz7XeFHLR5Cre6KrZGB5ScUdc1yGQZjhiZOR09jgTKIbJNPEzeGsAV2rJg9qUiOWLapz1XypdfkJh7MCNSKFINtxdA4I+NSDnuhER2kUi7K2HzrOLoLyXnUyuIxrkFf68sezDEhI1G911Q4f6GrqPkh1X201RSHW9rXQsKhXGHhoPl8UEuFcnLzALpukJ5LXg+V9kfwwmsmNxkV+/fKCVcZ1L3p/yCTAvZqmKXjTemYvELIuEWHYGVRdGJDadagYFqwuLMhEpogbJW+gZomT4vnIkhh/OIlFzeYTpKcQmqkIFqlwivZReiPTb19euk46n/n2pLo7dimu8rDvJwbJPimzlOf8bhOThEU9F5HIKLZlQR1+KCufaIszJ6SKAkwC9MQA+GYGnpoVbqE3ID84lypDaNS8CQZxEhwhoHuXZhl1oiZmec6YSRDsBX16NDsVvzDalrmwaBeHCwbyCd3A4490OMtydpPUU4MLQd/Qp4Gu4pZ2cp7OemMRes/PPF2rRdMZq2r/rri2KANY1pBf4m4QPqyRAscArk0Vn+2NS1X2Ede7o87DfvE8NDkbtm29LJA3PtX4fsnVwexAG/b/LBRk95uRVZBT6+dJ1jPtrpJKkFgZaMgevhxjigqvAqckfgmj86plkEdRRmeGjeReiwkCnE0Bf+sCoYt9HjkO0CEujwtO4aGeTSNlwz4/2rzZmxrCUdMuLzX2Wt+sppyvRFspbkLvYNtd+mQJNXdWKyjnKdNeckdHS1ZcuijQWpY1X7ifGLUPPSNUWZF8USLp3qU10Ko22zUX9Xhif9I3mgj0HcJk/ueKqxl5JqkFOB0b4V9ltdIWMWCkfszLqRAeh6FCTpoNVAU3ZF+k2diPtYhtAGwvDHOmZ+MfYqrddfoUqxlA0eCHv2FZ35bJmNxN0PGVvHMJNvDIjJ33xIlhR8v6KExeGBoczJUbB5pjjD+jzx99/XPw4u/zx/Y8ff7z4txIhvNsFfZfG5B7xCZiezMP/7HcosfDVaGBGPpbINDbiP3Zl1w3lqhEf1RS7MqhcBlqHMxXiHHLEV8J3HZsI4to3y8mfLaG2xoVfI7ndOLRLozkZEcrHFqNAOe46lnNYfrcShgeDag3gN8m28bu7Td/BMuxxk7p76q+VsrRv6YoHp3GHqMtxoDIYWndU3gRQ/4LF9UUjTTWqH5v2KyaWqNAUTqvbRvg2+g/YjbiOOZHUrMl8kaiJ7JTgnJ2Pic7VsxABRylPgE0ElhCzhqG2J6GEYGEeBlH4J/KtRnPkmyg5M+13eS9SEE48h93K6AIDyrEPxwUypHRgTFayDprDzBvLIrEQOW282glWCZ6IH8JZGPSyRVifZdJSVVAKZChvmj3gASPVVo/P3elhglPXW7Vdo2ACxso12EfpqodxdLiJYUYWS5ec3VxaMnZk84A8zlwcYvgnJ668xfvc3LZutHcODUpqV5NkOR6S0X1LNVfNoDYnRTXI1Xr21g1mt3mZA+qpFrcM033QH3nD+VrlfHrjiCDFME+SfDLasXw/BSNFKmSKXpk9XuBdOjU2otrNP8fO/rDfmMcPQ0mLYX+wd6hAkMY+kYaoCITDBxpgZ8gRO++XOU3l5fahYEARUSl/OYqaaWIDJuUrUiXJPGaWSlDMGhR+hvcbzEs+EQHH0VH58iOIrz5OUenSgzrXis+DElfwk/YtIBbQCOSJAixOO7G6serihzJL1UVhQ6xSadHogmv1MFJZWChknyKgjaNuXWcytAWM5WMkBz6n8pPSMzKMlULcGbW8td+DzF8rbct0txOpkdAgCOyAYdVUnVWVmeY68w1wNtJJmAyr3pUFaoU1bD/5qCPSIJfWeAVThcg1q4h7GvHeKjGp+Qz6K1LuBmVy6+S3aboAW3OUTrdWKNOGAQw3GbOqmgYI064ss/uk579Z+UcXRaqFg1NM1msbm0qFLotd6bo3zRXa11i2YCwwJUFvuxsKU0tXk4GrhNNSp9FUMZzLulYVH1NWwuai7HGcy3dn1K5e3mgiMPwr2vaTBQGcfZkKwLm66epcqlsOQOVDbNn5rXlLtOjZziQ2tEGyamVvmvfPP8esPU+UyhR7lGNGDk4zXUq4pl4bEbh+S4Qlt1aAN29L3aqxqxLNOIKZIFEXTm9uYQOBM+FwZVcb4GnwNPuvjl0uvbxUgzFPXUCmjqSqtGLcxfc3IMYFK1ybMALeh/NQX3uoLrHaMCRWT4GEuACO5k9RQq6m3t1whGxoVypeS61P3EC4boUvkxtvWI/mw0lXfl4qknyNoWHTlVsGEAUTEdHNlFiyQlJB9ubwm8pHSbiA3nbbCRa3jFfnaZZrAxpoZvrqqbNhFXaRnR4wO33jPFEf642rbjF8VL5xlhv6LDlSYK8P+i3A/yXALyi3Vnmb5KjZVIdpo9tHZ3LM7Xx1g83JUhJ3G3W+prmJ5SBSrgu8++pJ8lI0pL4p2kgqTQiibhrFLLYtApRU1/8cZOJU3bLZ7JKF20CxnU3SaSurwm4TbNRpmBQJge69bY2tGM/NRJ3Z/R7bCLW1tLxemjJbQr9dfaeba7AfpOLGr1VWRqPZBTsj7DKb4Tl8wQralmxJtQWhBk0YJbO0az1R7YoqtUm0FmAzA6zLtdt6Ni9b2YBlbXnVvoTfY/Oxe2gz+nwHHKbkuYlG/+CcxdOomAkWrDHLFTA8PVQSYoArHA3BbZyg5E13+6moBIqU/PfZeebVm2z0QqpkpXdr3Bsdqz2sISOElB5IVZwJDOoPNuwJed1KizC+8XJXnfi+FOv5lhOgl8N+5Sqw7lYKwGTyn44arAE7XTt7ixkbjBZNq5vkg4lwSfObwRaQT7I0QUgLnPZRBn7dygRF27BmK9GNNZpLyuqcghXnE9spzSi1ZEN0VKlMte27kNu7skwoN3ROqMcNxbWJ5Ube29z22ZgezJq7322xRhdxpzQZdl6+5Pa6toXTmHdTXy7zf1Tcii1XhpKlq62gYuwEAJzZObU1Zy8VNLJIeRcpjl6+3EZRz1Cs2tn5UqtIX4BcSxBeML9FsX/2Umt+mwKPuVrDNbKsa+HRyAo3LcfH8TQvMFAbSwEEfHX9i+odtC9uSGmZvXH0NaqOvLW1rNR6jauqXTtnpHm2Kk+NnQ2X9o7aPIfwNu8vn84+/Tri26UcuucVNyDm+VNeH6iyZF2r56ADCcsJYiZ1CJQaP1jxtXpKzPCaAKe0DjTwtgtmbzaASr4rGI2DwBB0mPwHgeMYjHJ+KfyCbm2fCnZkWQoKAuGVfgTiHMxmVTenKFhxhjhbWFMZFflzTUdCXDp+0WoFkL7iIsxXvjJs++ym3Oilowb3gYo4QYqHWqPyuCMxA1oAGLhPmEdZYmXsuYawFnULYjaS14hKimrSgCCd3oUPNQJgQmyaMiu0uSlKUi55adisXOdYJkojH52yccNQukX+eIbbQWUjRthUIlVMb4JQZE2BN2z54tNwEdwLX08FcJLlEJHJ/JMDm9HhgiIoy6/Q7M76/wGkogK0"
for relative, contents in json.loads(zlib.decompress(base64.b64decode(payload))).items():
    destination = support / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(".writing")
    temporary.write_text(contents)
    os.replace(temporary, destination)
for name in list(sys.modules):
    if name == "kaggle_yue2" or name.startswith("kaggle_yue2."):
        del sys.modules[name]
if str(support) not in sys.path:
    sys.path.insert(0, str(support))
from kaggle_yue2.runtime import KaggleYuE2, SongConfig
print("Notebook support loaded. No model has been executed yet.")

**Install and check Kaggle.** The isolated environment inherits Kaggle's
CUDA/PyTorch and scientific stack, with constraints preventing their upgrade.
A dependency conflict stops installation rather than replacing that stack.
The package list and hardware report are saved for diagnosis.

The checkpoint is about 4 GB. Setup requires at least 12 GiB of free disk.
To avoid downloading it again, set `LOCAL_CHECKPOINT` to the official file
attached under `/kaggle/input/...`; its size and checksum are still verified.

In [ ]:
ROOT = "/kaggle/working/yue2"
GPU_INDEX = 0
LOCAL_CHECKPOINT = None  # Or "/kaggle/input/your-dataset/yue2_3b_int8_convrot.safetensors"
if "app" in globals():
    app.stop()
app = KaggleYuE2(ROOT, gpu=GPU_INDEX)
runtime = app.install(local_checkpoint=LOCAL_CHECKPOINT)

**Song settings.** Replace these example lyrics with the complete song.
A seven/eight-minute composition needs enough material for verses, returns,
instrumental development, a bridge and an outro. Repeated lines alone do
not establish full-song quality. Planning budgets do not enforce duration.

Start at 30 seconds to verify the stack; a cutoff in this short diagnostic
is expected if the model has not reached an ending. Use `420` or `480`
only after that run works. Use `PLAN_ONLY = True` to review/edit ABC first.
A truncated ABC score fails explicitly and leaves its token artifact on disk.
For an edited/supplied score, set `CUSTOM_ABC` and keep the original run.

In [ ]:
MAX_DURATION_SECONDS = 30  # Diagnostic=30; three minutes=180; seven=420; eight=480
PLANNING_MODE = "full"    # "full", "melody", or "off"
PLAN_ONLY = False
CUSTOM_ABC = ""
MAX_ABC_TOKENS = 4096
ACOUSTIC_STEPS = 32       # Native dpm_2 / sgm_uniform baseline
SEMANTIC_CFG = 1.0       # Raising this adds a second AR guidance branch
ABC_SEED, SEMANTIC_SEED, ACOUSTIC_SEED = 10, 17, 7
STYLE = "English melodic rock, warm expressive lead vocal, electric guitar, piano, live drums, 112 BPM"
LYRICS = """[Verse 1]
Morning breaks across the water
Every road is turning gold
We have miles of open country
We have stories still untold

[Chorus]
Carry the light, carry it home
Across the night, we are not alone
Follow the river, follow the sun
The journey has only begun

[Verse 2]
Through the rain we kept on walking
Through the wind we found a song
Every voice became a lantern
Every step has brought us on

[Chorus]
Carry the light, carry it home
Across the night, we are not alone
Follow the river, follow the sun
The journey has only begun

[Bridge]
When the road bends out of sight
We will find another light

[Final Chorus]
Carry the light, carry it home
Across the night, we are not alone
Follow the river, follow the sun
The journey has only begun

[Outro]
Carry it home
We are not alone
"""
song = SongConfig(
    style=STYLE, lyrics=LYRICS, max_duration=MAX_DURATION_SECONDS,
    mode=PLANNING_MODE, abc=CUSTOM_ABC, plan_only=PLAN_ONLY,
    max_abc_tokens=MAX_ABC_TOKENS, steps=ACOUSTIC_STEPS,
    semantic_cfg=SEMANTIC_CFG, abc_seed=ABC_SEED,
    semantic_seed=SEMANTIC_SEED, acoustic_seed=ACOUSTIC_SEED,
)
print(f"Maximum: {song.max_duration / 60:.2f} minutes; planning: {song.mode}")

**Generate.** This uses a private local API, so there is no tunnel to lose.
Keep this cell running. Stage status is printed as it changes; detailed
model progress goes to `server.log`. A run can take substantially longer
than the audio on T4; no speed claim has been established.

To stop: interrupt this cell. It sends an interrupt to the managed server.
Cancellation takes effect at native processing boundaries; wait for the
server to become idle before another request. `app.stop()` terminates only
this notebook's process if a GPU operation is stuck. Partial artifacts remain.

In [ ]:
app.start()
run_dir = app.run(song, timeout=2 * 60 * 60)

**Inspect the result.** `summary.json` distinguishes natural endings from
cutoffs and reports actual audio length. Natural ending means the model
emitted its end token; it does not prove lyrical or musical completeness.
Listen to the outro and every listed acoustic boundary. Seven/eight-minute
generations may have multiple independently synthesized acoustic chunks.
The native ComfyUI decoder also applies its standard loudness adjustment;
any additional PCM export gain is recorded explicitly.

In [ ]:
from IPython.display import Audio, display
print((run_dir / "summary.json").read_text())
if (run_dir / "audio.wav").exists():
    if json.loads((run_dir / "summary.json").read_text())["audio"]["duration_seconds"] <= 180:
        display(Audio(filename=str(run_dir / "audio.wav")))
    else:
        print("Long audio: download audio.flac/audio.wav from Kaggle's Output files panel.")
elif (run_dir / "score.abc").exists():
    print((run_dir / "score.abc").read_text())

**Recovery (optional).** A failed VAE decode can reuse `latents.safetensors`
without rerunning the score, semantic generation or acoustic sampler.
For a sampler failure, `RECOVERY_MODE = "tokens"` reuses completed token
stages and recomputes acoustic synthesis. Recovery checks inputs, runtime
identities and artifact checksums; it fails on a mismatch instead of silently
reusing the wrong song. Each attempt gets a new run directory.

For a previous-session ZIP, extract it under a new directory in
`/kaggle/working`, or attach its extracted files as a Kaggle input. Point
`RECOVER_RUN` at the folder containing `request.json`.

In [ ]:
RECOVER_RUN = None  # Example: "/kaggle/working/yue2/runs/20261023T120000Z-0123456789"
RECOVERY_MODE = "decode"  # "decode" or "tokens"
if RECOVER_RUN:
    app.start()
    if RECOVERY_MODE == "decode":
        run_dir = app.decode_saved(RECOVER_RUN, tile_size=256)
    elif RECOVERY_MODE == "tokens":
        previous = json.loads((Path(RECOVER_RUN) / "request.json").read_text())
        run_dir = app.run(SongConfig(**previous["song"]), resume_from=RECOVER_RUN)
    else:
        raise ValueError("Choose decode or tokens")

**Preserve the run and release GPU memory.** Set `MAKE_ZIP = True` to package
the selected run. Download it through Kaggle's Output files panel or save
a notebook version with outputs. Do not assume live working files persist
when a session ends. Setup caches and dependencies remain on disk; the
managed process is stopped below. To generate again, rerun the settings
and generation cells; the latter restarts the managed process.

In [ ]:
MAKE_ZIP = False
if MAKE_ZIP and "run_dir" in globals():
    print("Download this archive from Kaggle Output:", app.archive(run_dir))
app.stop()
print("Managed server stopped; generated files remain under", app.runs)